# Try `google/t5-efficient-tiny` for Resume-to-JSON

This notebook is designed to run in Google Colab. It downloads the small T5 checkpoint, runs a baseline resume-to-JSON generation, and validates the result.

> Important: the base checkpoint is not fine-tuned for resume parsing. This is a smoke test of the model and prompt. Good production accuracy requires supervised resume-text-to-JSON examples.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers sentencepiece accelerate pydantic safetensors pypdf

## 2. Load the tiny model

In [ ]:
import json
import re
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/t5-efficient-tiny"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Loading: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")


## 3. Define the output contract

This is a compact validation contract for the first experiment. The full project schema remains in `src/schema/resume_output.py`.

In [ ]:
from typing import Any
from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class PersonalInformation(StrictModel):
    full_name: str | None = None
    professional_title: str | None = None
    email: str | None = None
    phone_numbers: list[str] = Field(default_factory=list)
    linkedin: str | None = None
    portfolio: str | None = None

class WorkExperience(StrictModel):
    job_title: str | None = None
    company: str | None = None
    location: str | None = None
    start_date: str | None = None
    end_date: str | None = None
    is_current: bool | None = None
    description: str | None = None
    responsibilities: list[str] = Field(default_factory=list)
    achievements: list[str] = Field(default_factory=list)
    skills_used: list[str] = Field(default_factory=list)

class Education(StrictModel):
    institution: str | None = None
    degree: str | None = None
    field_of_study: str | None = None
    start_date: str | None = None
    end_date: str | None = None
    graduation_date: str | None = None
    gpa: str | None = None

class Skills(StrictModel):
    technical: list[str] = Field(default_factory=list)
    tools_and_software: list[str] = Field(default_factory=list)
    domain: list[str] = Field(default_factory=list)
    soft: list[str] = Field(default_factory=list)

class ResumeOutput(StrictModel):
    personal_information: PersonalInformation = Field(default_factory=PersonalInformation)
    professional_summary: str | None = None
    career_objective: str | None = None
    target_roles: list[str] = Field(default_factory=list)
    work_experience: list[WorkExperience] = Field(default_factory=list)
    education: list[Education] = Field(default_factory=list)
    skills: Skills = Field(default_factory=Skills)
    certifications: list[dict[str, Any]] = Field(default_factory=list)
    projects: list[dict[str, Any]] = Field(default_factory=list)
    awards_and_honors: list[dict[str, Any]] = Field(default_factory=list)
    publications: list[dict[str, Any]] = Field(default_factory=list)
    volunteer_experience: list[dict[str, Any]] = Field(default_factory=list)
    professional_memberships: list[dict[str, Any]] = Field(default_factory=list)
    references: list[dict[str, Any]] = Field(default_factory=list)
    additional_sections: list[dict[str, Any]] = Field(default_factory=list)

print("Output contract loaded.")

## 4. Prepare a sample resume

In [ ]:
sample_resume = """
JANE DOE
Senior Data Analyst
jane.doe@example.com | +1 555 123 4567 | New York, NY
linkedin.com/in/janedoe

PROFESSIONAL SUMMARY
Data analyst with 6 years of experience turning business data into reporting and product insights.

EXPERIENCE
Senior Data Analyst — Acme Retail, New York, NY | March 2021 - Present
Built dashboards in SQL, Python, and Tableau. Automated weekly reporting and reduced manual work by 40 percent.

Data Analyst — Northwind Labs, Boston, MA | June 2018 - February 2021
Analyzed customer behavior and presented recommendations to product managers.

EDUCATION
Bachelor of Science in Statistics, Boston University, 2018

SKILLS
Python, SQL, Tableau, Excel, pandas, data visualization, experimentation, communication
""".strip()
print(sample_resume)

## 5. Generate JSON

T5 has a short context window, so we cap the input at 512 tokens for this first smoke test. Full resumes will need section chunking and merging.

In [ ]:
def build_prompt(resume_text: str) -> str:
    return (
        "extract resume information as JSON. return JSON only. "
        "Use these top-level keys: personal_information, professional_summary, "
        "career_objective, target_roles, work_experience, education, skills, "
        "certifications, projects, awards_and_honors, publications, "
        "volunteer_experience, professional_memberships, references, "
        "additional_sections. Use null for unknown single values and [] for "
        "unknown lists. resume: " + resume_text
    )

def generate_resume_json(resume_text: str, max_input_tokens: int = 512) -> str:
    prompt = build_prompt(resume_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    ).to(DEVICE)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=384,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

raw_output = generate_resume_json(sample_resume)
print(raw_output)

## 6. Parse and validate the response

In [ ]:
def parse_json_object(text: str) -> dict:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.IGNORECASE)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("The model did not return a JSON object.")
    parsed, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(parsed, dict):
        raise ValueError("The model output is not a JSON object.")
    return parsed

try:
    parsed_output = parse_json_object(raw_output)
    validated_output = ResumeOutput.model_validate(parsed_output)
    print("VALID JSON and schema-compatible output")
    print(json.dumps(validated_output.model_dump(), indent=2))
except Exception as error:
    print("Validation failed:", error)
    print("This is expected for an unfine-tuned base checkpoint.")

## 7. Optional: upload your own TXT or text-based PDF

For image-only PDFs, OCR must be run first. This cell handles `.txt` files and PDFs that already contain selectable text.

In [ ]:
from google.colab import files
from pathlib import Path
from pypdf import PdfReader

uploaded = files.upload()
uploaded_name = next(iter(uploaded))
uploaded_path = Path(uploaded_name)

if uploaded_path.suffix.lower() == ".txt":
    uploaded_resume = uploaded_path.read_text(errors="ignore")
elif uploaded_path.suffix.lower() == ".pdf":
    reader = PdfReader(str(uploaded_path))
    uploaded_resume = "\n".join(page.extract_text() or "" for page in reader.pages)
else:
    raise ValueError("Upload a .txt file or a text-based .pdf file.")

print(uploaded_resume[:3000])
uploaded_raw_output = generate_resume_json(uploaded_resume)
print("\nMODEL OUTPUT:\n", uploaded_raw_output)

## Interpretation

If the output is incomplete or invalid, that does not mean the experiment failed. It confirms that the base checkpoint needs supervised fine-tuning. The next dataset format should be:

```text
input:  cleaned OCR resume text
target: canonical ResumeOutput JSON
```

For a fair test, create a small set of manually reviewed input-target pairs, fine-tune the model, and then measure JSON validity plus field-level precision and recall.